# Session 5a : Concurrence et parallélisme en Python

#### Master TIDE 2026-2027

0. Sequentiel
1. Multiprocessing
2. Multithreading
3. Exercices

&copy; Francis Wolinski 2026

On considère la fonction `f` suivante où un `time.sleep(1)` simule une attente d’I/O par exemple :

```python
def f(name):
    for i in range(10):
        print(f"{i} : {name}")
        time.sleep(1)
```

## 0. Séquentiel

L’exécution **séquentielle** est le mode par défaut de Python : les instructions, et donc les appels de fonction, s’exécutent l’une après l’autre, dans l’ordre, sur un seul cœur du processeur. Un appel ne démarre que lorsque le précédent est terminé, ce qui rend le code simple à écrire, à lire et à déboguer, sans aucun risque de conflit entre les tâches. En revanche, pendant qu’une tâche attend (une lecture de fichier, une réponse réseau, un time.sleep) ou effectue un long calcul, rien d’autre ne progresse : les durées s’additionnent.

Dans le script `exemple_sequentiel.py`, l'instruction `list(map(f, ["Alice", "Bob"]))` exécute d’abord `f("Alice")` en entier, puis `f("Bob")`, ce qui donne un temps d’environ 20 secondes sur cette séquence, qui servira de référence pour comparer les approches parallèles.

```python
def run():
    list(map(f, ["Alice", "Bob"]))
```

## 1. Multiprocessing

Le **multiprocessing** consiste à lancer plusieurs processus indépendants, chacun avec son propre interpréteur Python et sa propre mémoire. Comme chaque processus tourne séparément, le *Global Interpreter Lock* (GIL) de CPython ne les bloque pas : ils peuvent s’exécuter réellement en parallèle sur plusieurs cœurs du processeur. C’est donc la solution adaptée aux tâches dîtes *CPU-bound* (calculs numériques intensifs, transformation de gros jeux de données, entraînement de modèles). En contrepartie, les processus sont plus coûteux à démarrer et ne partagent pas leur mémoire : les données doivent être échangées par sérialisation (par ex., **json**, **pickle**).

**CPU-Bound Tasks**: Operations that require significant CPU time for computations, like complex calculations, data processing, or image manipulation. Voir : https://medium.com/@shane-zhang/demystifying-python-concurrency-io-bound-vs-cpu-bound-tasks-64016db696c7

Dans le script `exemple_multiprocessing.py`, `Pool.map()` répartit automatiquement les appels de la fonction entre plusieurs processus.

```python
from multiprocessing import Pool

def run():
    with Pool() as p:
        p.map(f, ["Alice", "Bob"])
```

## 2. Multithreading

Le **multithreading** lance plusieurs threads (fils d’exécution) au sein d’un même processus : ils partagent la même mémoire, ce qui les rend légers et rapides à créer. En revanche, à cause du GIL, un seul thread exécute du code Python à la fois : le multithreading n’accélère donc pas les calculs, mais il est très efficace pour les tâches dîtes *I/O-bound* (lecture de fichiers, requêtes réseau, appels à une API), où le programme passe son temps à attendre et où un autre thread peut prendre le relais pendant ces temps d'attente. On peut gérer les threads « à la main » (création, `start()` puis `join()`) ou, plus simplement, avec `ThreadPoolExecutor`, qui s’occupe de tout.

**IO-Bound Tasks**: Operations that spend most of their time waiting for input/output operations to complete, such as reading from disk, network requests, or database queries. Voir : https://medium.com/@shane-zhang/demystifying-python-concurrency-io-bound-vs-cpu-bound-tasks-64016db696c7

Dans les scripts `exemple_multithreading0.py` et `exemple_multithreading.py`, les deux threads s’alternent et le temps total est divisé par deux par rapport à l’exécution séquentielle.

```python
from multiprocessing import Pool

def run():
    with ThreadPoolExecutor() as executor:
        executor.map(f, ["Alice", "Bob"])
```

## 3. Exercices

### a) Exercice de calculs sur les nombres premiers (*CPU-bound*)

On considère une liste de "grands" nombres premiers.

Adapter le code des différents scripts (séquentiel, multiprocessing, multithreading) pour tester leur primalité.

```python
numbers = [
    999_999_999_989,
    999_999_999_959,
    999_999_999_961,
    999_999_999_877,
    999_999_999_863,
    1_000_000_000_039,
    1_000_000_000_061,
    1_000_000_000_063,
    1_000_000_000_091,
    1_000_000_000_121,
    ]

def f(n):
    is_prime = n > 1 and all(n % d for d in range(2, int(sqrt(n)) + 1))
    print(n, is_prime)
```

### b) Exercice de webscraping d'URLs avec requests + pandas (*IO-bound*)

On considère un ensemble d'URLs contenant des tables HTML.

On utilise `pandas.read_html()` pour collecter les tables.

Comme Wikipedia bloque l'accès direct, on utilise par la librairie **requests** en passant un `User-Agent` approprié comme ci-dessous.

Adapter le code des différents scripts (séquentiel, multiprocessing, multithreading) pour collecter les tables HTML des différentes URLs données.

```python
url1 = "https://fr.wikipedia.org/wiki/Produit_int%C3%A9rieur_brut_par_habitant"
url2 = "https://fr.wikipedia.org/wiki/Jeux_olympiques"
url3 = "https://fr.wikipedia.org/wiki/Langues_les_plus_parl%C3%A9es"

urls = [url1, url2, url3]

def f(url):
    email_adress = "prenom.nom@gmail.com"  # à compléter
    headers = {"User-Agent": f"Mozilla/5.0 (cours data science; {email_adress})"}
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    tables = pd.read_html(StringIO(response.text))
    print([df.shape for df in tables])
```